# DarwinBot: train on Kaggle (CPU)

Runs `experiments/train.py` on a Kaggle CPU session (4 cores, up to 12 h), so
your own computer stays free.

1. **Settings**: Accelerator *None*, Internet *On*, Persistence *Files only*.
2. Edit the settings cell, then **Save Version > Save & Run All (Commit)**: the
   session runs with the browser closed.
3. The archive and log land in the notebook **Output** (`checkpoints/`). The run
   saves after every batch, so stopping at the time limit loses at most the
   batch in progress.
4. **To continue in a new session**: add this notebook's output as an input
   (*Add Input > Your Work*), run again; the last `archive.pkl` found under
   `/kaggle/input` is resumed. Download `checkpoints/archive.pkl` to use the
   robots locally (`replay.py`, `export.py`).

In [ ]:
# Clone DarwinBot into /tmp (not saved with the notebook output) and install it.
# While the repository is private, add a Kaggle secret named GITHUB_TOKEN
# (Add-ons > Secrets) holding a GitHub token with read access to it.
import os, subprocess
REPO = "mpena2099/darwinbot"
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
subprocess.run(["git", "clone", "-q", "--depth", "1", url, "/tmp/darwinbot"], check=True)
os.chdir("/tmp/darwinbot")
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
print("cores:", os.cpu_count())

In [ ]:
# Settings
TERRAINS = "terrain"   # a set (flat, terrain, all) or a list such as "rubble,soft"
BATCHES = 30           # batches to run in this session (more on a resume)
HOURS = 11.5           # stop before Kaggle's 12 h limit
TASK = "forward"       # forward or turn (ignored on a resume)

In [ ]:
import glob
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
previous = sorted(glob.glob("/kaggle/input/**/archive.pkl", recursive=True), key=os.path.getmtime)
args = ["python", "experiments/train.py", "--terrains", TERRAINS, "--task", TASK,
        "--batches", str(BATCHES), "--workers", str(os.cpu_count()),
        "--out", "/kaggle/working/checkpoints"]
if previous:
    args += ["--resume", previous[-1]]
    print("resuming", previous[-1])
cmd = f"timeout {int(HOURS * 3600)} {' '.join(args)} 2>&1 | tee /kaggle/working/checkpoints/train.log"
subprocess.run(["bash", "-c", cmd + " | grep --line-buffered -E '^(batch|\\[darwinbot\\]|Best|===)'"])

In [ ]:
# What the archive holds now
subprocess.run(["python", "experiments/replay.py", "--list",
                "--archive", "/kaggle/working/checkpoints/archive.pkl"])